# Copilot Audit Log — Direct Ingester (Fabric)

End-to-end ingestion notebook that **calls Microsoft Graph directly** from inside Fabric, parses the audit-log records on the fly, and writes a Delta table the **AI-in-One Dashboard** and **ValueLens** PBITs consume directly.

**No PowerShell, no intermediate CSVs, no separate parser.** Replaces the previous flow:

```
OLD:  PowerShell script → local CSV → manual move into Files/audit_raw/ → Copilot_Audit_Log_Parser.ipynb → Delta
NEW:  This notebook (Graph → parse → Delta)
```

**Output**: Lakehouse Delta table `Copilot_Interactions_Parsed` (same name + schema as the existing parser, so the PBIT works without changes).

**Schedule**: weekly (matches Microsoft Graph's 7-day audit-log query window). Use the **Schedule** button at the top of the notebook.

**Permissions**: app registration with `AuditLogsQuery.Read.All` (Application permission, admin-consented). No `Sites.Selected` needed — this path doesn't touch SharePoint.


## 1. Configuration

Fill in the four config values below. The client secret should ideally come from a Key Vault — see the commented alternative.

**Lakehouse Schemas note**: the default `OUTPUT_TABLE` is `dbo.Copilot_Interactions_Parsed`. This works for both schema-enabled Lakehouses (the new Fabric default) and non-schema Lakehouses — `dbo` is the implicit default schema either way. If your Lakehouse uses a different schema, change the prefix.


In [ ]:
# === CONFIG ===
TENANT_ID     = '<your-tenant-guid>'                  # Entra -> Overview -> Tenant ID
CLIENT_ID     = '<your-app-reg-client-id>'            # Entra -> App registrations -> your app -> Application (client) ID
CLIENT_SECRET = '<your-client-secret-value>'          # The secret VALUE (not Secret ID). Prefer Key Vault - see below.

# --- Load mode ---------------------------------------------------------------------
# 'backfill'    = pull BACKFILL_DAYS of history once (overwrite). Use for the initial load.
# 'incremental' = requery the trailing LOOKBACK_DAYS (for late arrivals) and merge by stable Id.
MODE          = 'incremental'
BACKFILL_DAYS = 180                                   # history to pull when MODE='backfill'
LOOKBACK_DAYS = 7                                     # trailing requery window + empty-table fallback
OUTPUT_TABLE  = 'dbo.Copilot_Interactions_Parsed'     # Delta table consumed by the PBIT (keep name/schema stable)

# --- Backfill scale tuning ---------------------------------------------------------
CHUNK_HOURS             = 8     # window size; backfill = many windows. 6-8h gives finer parallelism.
MAX_CONCURRENT_QUERIES  = 5     # Graph runs ~10 audit queries at once per tenant; 5 leaves room for other admins.
MAX_WAIT_MIN_PER_QUERY  = 240   # ceiling PER window (not the whole run). Resumable across reruns.
POLL_INTERVAL_SEC       = 30    # poll cadence per window
STAGING_DIR             = 'Files/_audit_staging'  # per-window JSONL + manifest; active runs only read their own windows.

# --- Failed-window recovery ----------------------------------------------------------
# A window whose query ends 'failed' or 'cancelled' (or times out) is sent again as a NEW query after a
# backoff. If it keeps failing it is split in half (8h -> 4h -> 2h -> 1h) and the halves are queried.
# Every window is attempted; the run only fails (without writing) if a window fails at MIN_CHUNK_HOURS.
WINDOW_RETRIES          = 3     # extra attempts per window before it is split
MIN_CHUNK_HOURS         = 1     # smallest size a failing window is split down to
SPLIT_ON_TIMEOUT        = True  # a window that hits MAX_WAIT_MIN_PER_QUERY splits at once instead of re-waiting
RETRY_BASE_SEC          = 60    # backoff before a retry: ~60s, 120s, 240s (with jitter) ...
RETRY_MAX_SEC           = 900   # ... never longer than this
MIN_CONCURRENT_QUERIES  = 1     # failures and HTTP 429s lower concurrency for the rest of the run, not below this
ALLOW_UNRECOVERED_GAPS  = False # incremental refuses to run while an older window failed and was never recovered

# --- Production: load secret from Key Vault instead of hardcoding -------------------
# from notebookutils.credentials import getSecret
# CLIENT_SECRET = getSecret('https://<your-vault>.vault.azure.net/', 'CopilotAuditAppSecret')


## 2. Authenticate to Microsoft Graph

Service principal (client secret) flow — same auth pattern as the PowerShell scripts.


In [ ]:
import requests, time
from requests.adapters import HTTPAdapter
try:
    from urllib3.util.retry import Retry
except Exception:
    from requests.packages.urllib3.util.retry import Retry

# Resilient HTTP session. The Microsoft Graph security/auditLog endpoints intermittently return
# 502/503/504 (gateway timeouts) and 429 (throttling), especially during the poll + records-fetch
# phases on busy tenants. A shared Session with urllib3 Retry transparently retries those with
# exponential backoff + Retry-After, so a transient blip no longer aborts the whole run.
THROTTLE_LISTENERS = []  # step 5 registers its query limiter here; called on every HTTP 429.


def _notify_throttle():
    for listener in list(THROTTLE_LISTENERS):
        try:
            listener()
        except Exception:
            pass


class _ThrottleAwareRetry(Retry):
    # urllib3 retries 429s before _request sees them; report each one so concurrency can back off.
    def increment(self, method=None, url=None, response=None, *args, **kwargs):
        if response is not None and getattr(response, 'status', None) == 429:
            _notify_throttle()
        return super().increment(method, url, response, *args, **kwargs)


def _build_session() -> requests.Session:
    s = requests.Session()
    retry = _ThrottleAwareRetry(
        total=8,                       # up to 8 attempts per request
        connect=5, read=5, status=8,
        backoff_factor=2,              # 0,2,4,8,16,32... seconds (capped by urllib3)
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset(['GET', 'POST']),
        respect_retry_after_header=True,
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry, pool_connections=8, pool_maxsize=8)
    s.mount('https://', adapter)
    s.mount('http://', adapter)
    return s

SESSION = _build_session()

# Belt-and-braces wrapper: urllib3 Retry handles most cases, but we add an outer retry loop so a
# request that *still* surfaces a transient status (or a connection error) after the adapter gives
# up is retried a few more times rather than killing the notebook.
_TRANSIENT = {429, 500, 502, 503, 504}

def _request(method: str, url: str, *, attempts: int = 5, **kwargs):
    kwargs.setdefault('timeout', 60)
    last = None
    for i in range(1, attempts + 1):
        try:
            r = SESSION.request(method, url, **kwargs)
        except requests.exceptions.RequestException as e:
            last = e
            if i == attempts:
                raise
            wait = min(60, 2 ** i)
            print(f'    transient network error ({type(e).__name__}); retry {i}/{attempts} in {wait}s')
            time.sleep(wait)
            continue
        if r.status_code == 429:
            _notify_throttle()
        if r.status_code in _TRANSIENT and i < attempts:
            wait = min(60, 2 ** i)
            print(f'    transient HTTP {r.status_code} on {method} {url.split("?")[0].rsplit("/",1)[-1]}; '
                  f'retry {i}/{attempts} in {wait}s')
            time.sleep(wait)
            continue
        return r
    return r  # pragma: no cover

# Auto-refreshing app-only Graph token. Long (multi-hour) chunked runs would otherwise
# fail when the 1-hour token expires mid-pull, so we transparently refresh every ~50 min.
def _get_graph_token() -> str:
    url  = f'https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token'
    body = {
        'client_id':     CLIENT_ID,
        'scope':         'https://graph.microsoft.com/.default',
        'client_secret': CLIENT_SECRET,
        'grant_type':    'client_credentials',
    }
    r = _request('POST', url, data=body, timeout=30)
    r.raise_for_status()
    return r.json()['access_token']

_token = {'value': None, 'ts': 0.0}

def get_headers() -> dict:
    if _token['value'] is None or (time.time() - _token['ts']) > 50 * 60:
        _token['value'] = _get_graph_token()
        _token['ts'] = time.time()
    return {'Authorization': f'Bearer {_token["value"]}', 'Content-Type': 'application/json'}

get_headers()
print('\u2713 Graph token acquired (auto-refreshing, retry-resilient session).')


## 3. Build query windows + the `create_query` helper

The lookback is split into smaller time windows (`CHUNK_HOURS`). Each window is its own async
audit query, which completes far faster than one giant query — the main cause of timeouts on
large tenants — and lets the run checkpoint window-by-window.

In [ ]:
from datetime import datetime, timezone, timedelta
import hashlib, json, random, threading, time

WINDOW_KEY_VERSION = 2
KEY_COLUMNS = ['Id', 'Source_RecordKey', 'Source_MessageKey', 'Source_ResourceKey']
_RESOURCE_KEY_FIELDS = ('Type', 'Action', 'SiteUrl', 'SensitivityLabelId')
_SYNTHETIC_AUDIT_FIELDS = frozenset({'_StableKey', '_StableOrdinal'})


def _as_utc_datetime(value):
    if value in (None, ''):
        return None
    if isinstance(value, datetime):
        dt = value
    else:
        text = str(value).strip()
        if not text:
            return None
        if text.endswith('Z'):
            text = text[:-1] + '+00:00'
        dt = datetime.fromisoformat(text)
    if dt.tzinfo is None:
        return dt.replace(tzinfo=timezone.utc)
    return dt.astimezone(timezone.utc)


def _norm_text(value) -> str:
    if value is None:
        return ''
    return ' '.join(str(value).strip().split())


def _coerce_json_value(value):
    if value is None:
        return None
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return None
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            return text
    return value


def _has_meaningful_value(value) -> bool:
    if isinstance(value, dict):
        return any(_has_meaningful_value(v) for k, v in value.items() if k not in _SYNTHETIC_AUDIT_FIELDS)
    if isinstance(value, list):
        return any(_has_meaningful_value(v) for v in value)
    if value is None:
        return False
    if isinstance(value, str):
        return _norm_text(value) != ''
    return True


def _canonical_json_text(value) -> str:
    return json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(',', ':'))


def _stable_message_base_key(message) -> str:
    message = dict(message or {})
    msg_id = _norm_text(message.get('Id'))
    if msg_id:
        return f'mid:{msg_id}'
    payload = {k: v for k, v in message.items() if k not in _SYNTHETIC_AUDIT_FIELDS and k != 'Id'}
    return 'msgf:' + hashlib.sha256(_canonical_json_text(payload).encode('utf-8')).hexdigest()


def _stable_resource_base_key(resource) -> str:
    resource = dict(resource or {})
    payload = {k: v for k, v in resource.items() if k not in _SYNTHETIC_AUDIT_FIELDS}
    if not _has_meaningful_value(payload):
        return 'resource:none'
    return 'resource:' + hashlib.sha256(_canonical_json_text(payload).encode('utf-8')).hexdigest()


def _stable_sequence(items, base_key_builder, path) -> list:
    prepared = []
    for item in items or []:
        canonical_item = _canonicalize_json(item, path)
        if not isinstance(canonical_item, dict):
            canonical_item = {'Value': canonical_item}
        canonical_text = _canonical_json_text(canonical_item)
        base_key = base_key_builder(canonical_item)
        prepared.append((base_key, canonical_text, canonical_item))
    prepared.sort(key=lambda entry: (entry[0], entry[1]))
    totals = {}
    for base_key, _canonical_text, _item in prepared:
        totals[base_key] = totals.get(base_key, 0) + 1
    seen = {}
    stable = []
    for ordinal, (base_key, _canonical_text, canonical_item) in enumerate(prepared):
        seen[base_key] = seen.get(base_key, 0) + 1
        stable_key = base_key if totals[base_key] == 1 else f'{base_key}#{seen[base_key]}'
        enriched = dict(canonical_item)
        enriched['_StableKey'] = stable_key
        enriched['_StableOrdinal'] = ordinal
        stable.append(enriched)
    return stable


def _canonicalize_json(value, path=()):
    value = _coerce_json_value(value) if not path else value
    if isinstance(value, dict):
        return {
            str(key): _canonicalize_json(val, path + (str(key),))
            for key, val in sorted(value.items(), key=lambda item: str(item[0]))
            if key not in _SYNTHETIC_AUDIT_FIELDS
        }
    if isinstance(value, list):
        if path == ('CopilotEventData', 'Messages'):
            return _stable_sequence(value, _stable_message_base_key, path)
        if path == ('CopilotEventData', 'AccessedResources'):
            return _stable_sequence(value, _stable_resource_base_key, path)
        return [_canonicalize_json(item, path) for item in value]
    return value


def canonicalize_audit_payload(audit_data):
    value = _coerce_json_value(audit_data)
    if value is None:
        return None
    return _canonicalize_json(value)


def determine_incremental_start(end_dt, high_water_mark, lookback_days: int):
    lookback_days = max(int(lookback_days), 0)
    lookback_floor = end_dt - timedelta(days=lookback_days)
    hw = _as_utc_datetime(high_water_mark)
    if hw is None:
        return lookback_floor
    return min(hw, lookback_floor)


def validate_incremental_target_columns(columns, required_columns, table_name: str):
    cols = set(columns or [])
    missing = [c for c in required_columns if c not in cols]
    if missing:
        raise RuntimeError(
            f"Legacy table {table_name} is missing stable key columns {missing}. "
            "Run one MODE='backfill' overwrite to upgrade the parsed audit table before incremental resumes."
        )


def build_source_record_key(
    record_id,
    creation_date=None,
    operation=None,
    audit_data=None,
    record_type=None,
    associated_admin_units=None,
    associated_admin_units_names=None,
) -> str:
    rid = _norm_text(record_id)
    if rid:
        return f"rid:{rid}"
    payload = {
        'CreationDate': _norm_text(creation_date),
        'Operation': _norm_text(operation),
        'RecordType': _norm_text(record_type),
        'AuditData': canonicalize_audit_payload(audit_data),
        'AssociatedAdminUnits': _canonicalize_json(associated_admin_units or []),
        'AssociatedAdminUnitsNames': _canonicalize_json(associated_admin_units_names or []),
    }
    return 'synthetic:' + hashlib.sha256(_canonical_json_text(payload).encode('utf-8')).hexdigest()


def canonicalize_audit_record(record) -> dict:
    record = record or {}
    audit_payload = canonicalize_audit_payload(record.get('auditData'))
    return {
        'RecordId': record.get('id'),
        'CreationDate': record.get('createdDateTime'),
        'RecordType': record.get('auditLogRecordType'),
        'Operation': record.get('operation'),
        'AuditData': _canonical_json_text(audit_payload) if audit_payload is not None else None,
        'SourceRecordKey': build_source_record_key(
            record.get('id'),
            record.get('createdDateTime'),
            record.get('operation'),
            audit_payload,
            record_type=record.get('auditLogRecordType'),
            associated_admin_units=record.get('associatedAdminUnits', []),
            associated_admin_units_names=record.get('associatedAdminUnitsNames', []),
        ),
        'AssociatedAdminUnits': _canonical_json_text(_canonicalize_json(record.get('associatedAdminUnits', []))),
        'AssociatedAdminUnitsNames': _canonical_json_text(_canonicalize_json(record.get('associatedAdminUnitsNames', []))),
    }


def stable_window_key(win_start, win_end) -> str:
    ws = _as_utc_datetime(win_start)
    we = _as_utc_datetime(win_end)
    return f"v{WINDOW_KEY_VERSION}_{ws:%Y%m%d%H%M}_{we:%Y%m%d%H%M}"


# --- Failed-window recovery: kept verbatim with shared/python/valuelens_core/audit.py -------------
# A split window keeps its own stable_window_key; its halves get theirs, so manifests written before
# splitting existed stay valid and every part is reusable on a rerun.
def split_window(win_start, win_end, min_hours):
    """Halves of a window for another try at a smaller size; [] when a half would be under min_hours."""
    ws = _as_utc_datetime(win_start)
    we = _as_utc_datetime(win_end)
    half_minutes = int((we - ws).total_seconds() // 60) // 2
    if half_minutes <= 0 or half_minutes < float(min_hours) * 60:
        return []
    mid = win_start + timedelta(minutes=half_minutes)
    return [(win_start, mid), (mid, win_end)]


def expand_split_windows(manifest, win_start, win_end, min_hours, include_parents=False):
    """The windows to query for one grid window: its halves (recursively) if an earlier run split it."""
    entry = (manifest or {}).get(stable_window_key(win_start, win_end))
    halves = []
    if isinstance(entry, dict) and entry.get('status') == 'split':
        halves = split_window(win_start, win_end, min_hours)
    if not halves:
        return [(win_start, win_end)]
    out = [(win_start, win_end)] if include_parents else []
    for ws, we in halves:
        out.extend(expand_split_windows(manifest, ws, we, min_hours, include_parents))
    return out


def retry_delay(attempt, base_seconds, max_seconds, rand=random.random):
    """Exponential backoff with jitter: base, 2x base, 4x base ... capped at max_seconds, times 50-100%."""
    raw = min(float(max_seconds), float(base_seconds) * (2 ** max(int(attempt) - 1, 0)))
    return max(0.0, raw * (0.5 + 0.5 * rand()))


class AdaptiveLimiter:
    """Caps concurrent audit queries. Failures and throttling lower the cap for the rest of the run.

    A burst of failures with one cause (several windows failing in the same second) counts once:
    the cap moves at most once per cooldown_seconds.
    """

    def __init__(self, limit, minimum=1, cooldown_seconds=60, clock=time.monotonic):
        self.minimum = max(1, int(minimum))
        self.limit = max(self.minimum, int(limit))
        self.active = 0
        self.history = []
        self._cooldown = float(cooldown_seconds)
        self._clock = clock
        self._last_shrink = None
        self._cond = threading.Condition()

    def __enter__(self):
        with self._cond:
            while self.active >= self.limit:
                self._cond.wait()
            self.active += 1
        return self

    def __exit__(self, *exc):
        with self._cond:
            self.active -= 1
            self._cond.notify_all()
        return False

    def shrink(self, reason, halve=False):
        with self._cond:
            now = self._clock()
            if self._last_shrink is not None and now - self._last_shrink < self._cooldown:
                return False
            new = max(self.minimum, self.limit // 2 if halve else self.limit - 1)
            if new >= self.limit:
                return False
            self.history.append((self.limit, new, reason))
            self.limit, self._last_shrink = new, now
            return True


def _manifest_range(entry):
    if not isinstance(entry, dict):
        return None
    try:
        ws = _as_utc_datetime(entry.get('window_start'))
        we = _as_utc_datetime(entry.get('window_end'))
    except (TypeError, ValueError):
        return None
    return (ws, we) if ws and we and ws < we else None


def uncovered_failed_ranges(manifest, not_before, before):
    """Time ranges in [not_before, before) of unfinished windows that no succeeded window covers."""
    lo = _as_utc_datetime(not_before)
    hi = _as_utc_datetime(before)
    entries = [e for e in (manifest or {}).values() if isinstance(e, dict)]
    covered = sorted(r for r in (_manifest_range(e) for e in entries if e.get('status') == 'succeeded') if r)
    gaps = []
    for entry in entries:
        if entry.get('status') in ('succeeded', 'split'):
            continue
        rng = _manifest_range(entry)
        if not rng:
            continue
        pieces = [(max(rng[0], lo), min(rng[1], hi))]
        for cs, ce in covered:
            pieces = [p for s, e in pieces for p in ((s, min(e, cs)), (max(s, ce), e)) if p[0] < p[1]]
        gaps.extend(pieces)
    merged = []
    for s, e in sorted(gaps):
        if merged and s <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(merged[-1][1], e))
        else:
            merged.append((s, e))
    return merged


if MODE not in ('backfill', 'incremental'):
    raise ValueError(f"Unsupported MODE={MODE!r}; expected 'backfill' or 'incremental'.")

end_date = datetime.now(timezone.utc)
if MODE == 'backfill':
    start_date = end_date - timedelta(days=BACKFILL_DAYS)
    WRITE_MODE = 'overwrite'
else:
    WRITE_MODE = 'merge'
    _hw = None
    if spark.catalog.tableExists(OUTPUT_TABLE):
        from pyspark.sql import functions as F
        _existing = spark.table(OUTPUT_TABLE)
        validate_incremental_target_columns(_existing.columns, KEY_COLUMNS, OUTPUT_TABLE)
        _hw = _existing.agg(F.max('CreationDate').alias('m')).collect()[0]['m']
    start_date = determine_incremental_start(end_date, _hw, LOOKBACK_DAYS)
    if _hw:
        print(
            f'incremental: high-water mark = {_as_utc_datetime(_hw):%Y-%m-%d %H:%M}; '
            f'requerying trailing {LOOKBACK_DAYS}d from {start_date:%Y-%m-%d %H:%M}'
        )
    else:
        print(f'incremental: no prior data, defaulting to last {LOOKBACK_DAYS}d')

_anchor_hour = start_date.hour - (start_date.hour % CHUNK_HOURS)
_cur = start_date.replace(hour=_anchor_hour, minute=0, second=0, microsecond=0)
windows = []
while _cur < end_date:
    _nxt = min(_cur + timedelta(hours=CHUNK_HOURS), end_date)
    windows.append((_cur, _nxt))
    _cur = _nxt
print(f'MODE={MODE}: {start_date:%Y-%m-%d} -> {end_date:%Y-%m-%d}  =>  {len(windows)} window(s) of {CHUNK_HOURS}h, WRITE_MODE={WRITE_MODE}')


def create_query(win_start, win_end) -> str:
    body = {
        'displayName':         f'Copilot Interactions {win_start:%Y%m%d%H%M}-{win_end:%Y%m%d%H%M}',
        'filterStartDateTime': win_start.isoformat(),
        'filterEndDateTime':   win_end.isoformat(),
        'recordTypeFilters':   ['copilotInteraction'],
        'operationFilters':    [],
    }
    r = _request('POST', 'https://graph.microsoft.com/beta/security/auditLog/queries',
                 json=body, headers=get_headers(), timeout=30)
    r.raise_for_status()
    return r.json()['id']


## 4. Resumable wait helper

Polls an async query until it succeeds, with a **generous, configurable** ceiling
(`MAX_WAIT_MIN_PER_QUERY`) instead of a hard 30-minute cut-off. The token auto-refreshes while
waiting, so multi-hour jobs don't fail on expiry.

A query the service ends as `failed` or `cancelled` raises `AuditQueryFailed`. Step 5 retries it with a
fresh query, and splits the window if it keeps failing.

In [ ]:
import time


class AuditQueryFailed(RuntimeError):
    """The audit service ended a query as failed or cancelled; a new query for the window may succeed."""


def _read_query_status(query_id: str):
    r = _request(
        'GET',
        f'https://graph.microsoft.com/beta/security/auditLog/queries/{query_id}',
        headers=get_headers(),
        timeout=30,
    )
    r.raise_for_status()
    payload = r.json()
    if not isinstance(payload, dict):
        raise RuntimeError(
            f'Query {query_id} returned malformed status payload type: {type(payload).__name__}'
        )
    status = payload.get('status')
    if not isinstance(status, str) or not status.strip():
        raise RuntimeError(f'Query {query_id} returned missing/invalid status: {status!r}')
    return payload, status.strip().lower()


def wait_for_query(query_id: str, max_wait_min: int = MAX_WAIT_MIN_PER_QUERY):
    started = time.time()
    deadline = started + max_wait_min * 60
    polls = 0
    while time.time() < deadline:
        polls += 1
        payload, status = _read_query_status(query_id)
        if status == 'succeeded':
            return payload
        if status in ('failed', 'cancelled'):
            raise AuditQueryFailed(f'Query {query_id} ended with status: {status}')
        if status not in ('notstarted', 'running'):
            raise RuntimeError(f'Query {query_id} returned unexpected status: {status}')
        if polls % 4 == 0:
            mins = int((time.time() - started) // 60)
            print(f'    … still {status} ({mins}m elapsed, {polls} polls)')
        time.sleep(POLL_INTERVAL_SEC)
    raise TimeoutError(
        f'Query {query_id} did not succeed within {max_wait_min} min. '
        f'Increase MAX_WAIT_MIN_PER_QUERY or reduce CHUNK_HOURS.'
    )


## 5. Fetch records → stream to Lakehouse Files (chunked, bounded memory)

Each page of results is written straight to Lakehouse **Files** as JSON-lines, so the full
result set is **never** held in driver memory — the other main cause of OOM/timeouts on large
tenants. Set `WRITE_MODE = 'overwrite'` to clear the staging area first.

**Failed windows recover on their own.** On large tenants a query sometimes ends `failed` or `cancelled`,
often when several queries are submitted at once or someone else is running audit searches. Each failing
window is:

1. sent again as a **new** query, up to `WINDOW_RETRIES` times, with exponential backoff and jitter;
2. then **split in half** (8h → 4h → 2h → 1h, down to `MIN_CHUNK_HOURS`), and the halves are queried.
   The halves are recorded in the manifest, so a rerun reuses every half that already succeeded.

Each failure or HTTP 429 also lowers the number of concurrent queries for the rest of the run
(not below `MIN_CONCURRENT_QUERIES`). Every window is attempted before the run reports anything.
If a window still fails at `MIN_CHUNK_HOURS`, the cell stops with one message listing the failed windows
and **nothing is written**. Rerun the notebook (in `MODE='backfill'` if this was a backfill); succeeded
windows are reused, so only the failed ones are queried again. An incremental run refuses to start while
an older window is unrecovered, so switching modes can't silently leave a gap.

In [ ]:
import json, os, tempfile, threading, time
from urllib.parse import urlparse
from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, wait as wait_futures

# Defaults for config cells copied from an older version of this notebook.
WINDOW_RETRIES = int(globals().get('WINDOW_RETRIES', 3))
MIN_CHUNK_HOURS = float(globals().get('MIN_CHUNK_HOURS', 1))
SPLIT_ON_TIMEOUT = bool(globals().get('SPLIT_ON_TIMEOUT', True))
RETRY_BASE_SEC = float(globals().get('RETRY_BASE_SEC', 60))
RETRY_MAX_SEC = float(globals().get('RETRY_MAX_SEC', 900))
MIN_CONCURRENT_QUERIES = int(globals().get('MIN_CONCURRENT_QUERIES', 1))
ALLOW_UNRECOVERED_GAPS = bool(globals().get('ALLOW_UNRECOVERED_GAPS', False))

# Resumable, bounded-concurrency backfill. Each window's records stream to a JSONL file and its
# status is recorded in a manifest; rerun skips older succeeded windows, but recent LOOKBACK_DAYS
# windows are intentionally re-queried for late arrivals. Active runs only read the current window set.
# A failing window is retried with a new query, then split into halves (status 'split', parts=[keys]);
# the halves have their own manifest entries and staging files.

def _normalize_stage_dir(path: str) -> str:
    if not isinstance(path, str) or not path.strip():
        raise ValueError(f'Stage directory must be a non-empty string; found {path!r}.')
    text = path.strip()
    if text.startswith('abfss://'):
        return text.rstrip('/')
    normalized = text.replace('\\', '/')
    mount_prefix = '/lakehouse/default/'
    if normalized.startswith(mount_prefix):
        normalized = normalized[len(mount_prefix):]
    if normalized == 'Files' or normalized.startswith('Files/'):
        return normalized.rstrip('/')
    return os.path.abspath(text)


def _is_remote_stage_dir(path: str) -> bool:
    return isinstance(path, str) and (path == 'Files' or path.startswith('Files/') or path.startswith('abfss://'))


def _stage_path_join(base_dir: str, name: str) -> str:
    if _is_remote_stage_dir(base_dir):
        return base_dir.rstrip('/') + '/' + name.lstrip('/')
    return os.path.join(base_dir, name)


def _stage_basename(path: str) -> str:
    if not isinstance(path, str) or not path.strip():
        raise ValueError(f'Stage path must be a non-empty string; found {path!r}.')
    return path.rstrip('/').replace('\\', '/').split('/')[-1]


def _file_uri(path: str) -> str:
    return 'file:' + os.path.abspath(path)


def _driver_temp_root(base_dir: str) -> str:
    configured = globals().get('LOCAL_STAGE_TMP_DIR')
    if isinstance(configured, str) and configured.strip():
        root = os.path.abspath(configured)
    elif _is_remote_stage_dir(base_dir):
        root = os.path.abspath(os.path.join(tempfile.gettempdir(), 'valuelens_audit_stage'))
    else:
        root = os.path.abspath(base_dir)
    os.makedirs(root, exist_ok=True)
    return root


def _new_local_temp_path(base_dir: str, stem: str, suffix: str) -> str:
    fd, temp_path = tempfile.mkstemp(
        prefix=f'.{stem}.{os.getpid()}.{threading.get_ident()}.',
        suffix=suffix,
        dir=_driver_temp_root(base_dir),
    )
    os.close(fd)
    return temp_path


def _get_notebook_fs():
    fs = globals().get('NOTEBOOK_FS')
    if fs is None:
        try:
            import notebookutils
        except ImportError as e:
            raise RuntimeError('Remote staging paths require notebookutils.fs in the Fabric runtime.') from e
        fs = getattr(notebookutils, 'fs', None)
    if fs is None:
        raise RuntimeError('Remote staging paths require notebookutils.fs.')
    missing = [name for name in ('ls', 'exists', 'rm', 'cp', 'mv', 'mkdirs') if not hasattr(fs, name)]
    if missing:
        raise RuntimeError(f'notebookutils.fs is missing required method(s): {missing}.')
    return fs


def _ensure_stage_dir(base_dir: str):
    if _is_remote_stage_dir(base_dir):
        fs = _get_notebook_fs()
        if not fs.exists(base_dir):
            fs.mkdirs(base_dir)
        return
    os.makedirs(base_dir, exist_ok=True)


def _list_stage_paths(base_dir: str) -> list:
    if _is_remote_stage_dir(base_dir):
        fs = _get_notebook_fs()
        if not fs.exists(base_dir):
            return []
        paths = []
        for entry in fs.ls(base_dir):
            path = getattr(entry, 'path', None)
            if path is None and isinstance(entry, dict):
                path = entry.get('path')
            if not isinstance(path, str) or not path.strip():
                raise RuntimeError(f'Stage listing entry for {base_dir} is missing a usable path: {entry!r}')
            paths.append(path.rstrip('/'))
        return sorted(set(paths))
    if not os.path.isdir(base_dir):
        return []
    return sorted(os.path.join(base_dir, name) for name in os.listdir(base_dir))


def _stage_exists(path: str) -> bool:
    if _is_remote_stage_dir(path):
        return _get_notebook_fs().exists(path)
    return os.path.exists(path)


def _stage_remove(path: str):
    if _is_remote_stage_dir(path):
        fs = _get_notebook_fs()
        if fs.exists(path):
            if fs.rm(path, False) is False:
                raise RuntimeError(f'Failed to remove staged file {path}.')
        return
    if os.path.exists(path):
        os.remove(path)


def _write_stage_text(base_dir: str, target_path: str, text: str):
    _ensure_stage_dir(base_dir)
    local_temp = _new_local_temp_path(base_dir, _stage_basename(target_path), '.tmp')
    try:
        with open(local_temp, 'w', encoding='utf-8') as f:
            f.write(text)
            f.flush()
            os.fsync(f.fileno())
        if _is_remote_stage_dir(base_dir):
            fs = _get_notebook_fs()
            _stage_remove(target_path)
            copied = fs.cp(_file_uri(local_temp), target_path)
            if copied is False:
                raise RuntimeError(f'Failed to copy local staging file to {target_path}.')
        else:
            os.replace(local_temp, target_path)
            local_temp = None
    finally:
        if local_temp and os.path.exists(local_temp):
            os.remove(local_temp)


def _write_stage_jsonl(base_dir: str, target_path: str, records):
    _ensure_stage_dir(base_dir)
    local_temp = _new_local_temp_path(base_dir, _stage_basename(target_path), '.jsonl')
    try:
        with open(local_temp, 'w', encoding='utf-8') as f:
            for record in records:
                f.write(json.dumps(_row(record)) + '\n')
            f.flush()
            os.fsync(f.fileno())
        if _is_remote_stage_dir(base_dir):
            fs = _get_notebook_fs()
            _stage_remove(target_path)
            copied = fs.cp(_file_uri(local_temp), target_path)
            if copied is False:
                raise RuntimeError(f'Failed to copy local staging page to {target_path}.')
        else:
            os.replace(local_temp, target_path)
            local_temp = None
    finally:
        if local_temp and os.path.exists(local_temp):
            os.remove(local_temp)


def _move_stage_file(base_dir: str, source_path: str, target_path: str, overwrite: bool = True):
    if _is_remote_stage_dir(base_dir):
        fs = _get_notebook_fs()
        # Fabric's third argument is create_path, not overwrite. Rename in place
        # without first deleting the valid destination checkpoint.
        moved = fs.mv(source_path, target_path, False, overwrite)
        if moved is False:
            raise RuntimeError(f'Failed to move staged file {source_path} to {target_path}.')
        return
    if not overwrite and os.path.exists(target_path):
        raise FileExistsError(target_path)
    os.replace(source_path, target_path)


def _read_stage_json(base_dir: str, source_path: str):
    if _is_remote_stage_dir(base_dir):
        local_temp = _new_local_temp_path(base_dir, _stage_basename(source_path), '.json')
        try:
            copied = _get_notebook_fs().cp(source_path, _file_uri(local_temp))
            if copied is False:
                raise RuntimeError(f'Failed to copy staged file {source_path} to local scratch.')
            with open(local_temp, encoding='utf-8') as f:
                return json.load(f)
        finally:
            if os.path.exists(local_temp):
                os.remove(local_temp)
    with open(source_path, encoding='utf-8') as f:
        return json.load(f)


STAGING_ABS = _normalize_stage_dir(STAGING_DIR)
MANIFEST = _stage_path_join(STAGING_ABS, '_manifest.json')
MANIFEST_LOCK = threading.RLock()
_ensure_stage_dir(STAGING_ABS)


def list_window_files(base_dir: str, key: str, include_partial: bool = False):
    prefix = f'win_{key}_'
    matches = []
    for path in _list_stage_paths(base_dir):
        name = _stage_basename(path)
        if not name.startswith(prefix):
            continue
        if name.endswith('.jsonl') or (include_partial and name.endswith('.jsonl.partial')):
            matches.append(path)
    return sorted(set(matches))


def purge_window_files(base_dir: str, key: str, include_partial: bool = True) -> int:
    removed = 0
    for path in list_window_files(base_dir, key, include_partial=include_partial):
        if _stage_exists(path):
            _stage_remove(path)
            removed += 1
    return removed


def list_active_stage_files(base_dir: str, active_keys) -> list:
    files = []
    for key in sorted(set(active_keys or [])):
        files.extend(list_window_files(base_dir, key, include_partial=False))
    return sorted(files)


def _allowed_manifest_statuses():
    return {'querying', 'waiting', 'draining', 'succeeded', 'failed', 'split'}


def _validate_manifest_payload(data):
    if not isinstance(data, dict):
        raise RuntimeError(
            f'Manifest {MANIFEST} must be a JSON object keyed by window id; found {type(data).__name__}.'
        )
    allowed = _allowed_manifest_statuses()
    for key, entry in data.items():
        if not isinstance(key, str) or not key.strip():
            raise RuntimeError(f'Manifest {MANIFEST} has an invalid window key: {key!r}.')
        if not isinstance(entry, dict):
            raise RuntimeError(
                f'Manifest {MANIFEST} entry {key!r} must be a JSON object; found {type(entry).__name__}.'
            )
        status = entry.get('status')
        if not isinstance(status, str) or status not in allowed:
            raise RuntimeError(
                f"Manifest {MANIFEST} entry {key!r} has invalid status {status!r}; "
                f'expected one of {sorted(allowed)}.'
            )
        if status == 'split':
            parts = entry.get('parts')
            if not isinstance(parts, list) or not parts or not all(isinstance(p, str) and p.strip() for p in parts):
                raise RuntimeError(f'Manifest {MANIFEST} entry {key!r} is split but has no valid parts list.')
    return data


def _load_manifest():
    if not _stage_exists(MANIFEST):
        return {}
    try:
        data = _read_stage_json(STAGING_ABS, MANIFEST)
    except json.JSONDecodeError as e:
        raise RuntimeError(f'Manifest {MANIFEST} is not valid JSON: {e}') from e
    except UnicodeDecodeError as e:
        raise RuntimeError(f'Manifest {MANIFEST} is not valid UTF-8 JSON: {e}') from e
    return _validate_manifest_payload(data)


def _manifest_temp_path():
    base = _stage_basename(MANIFEST)
    return _stage_path_join(STAGING_ABS, f'.{base}.{os.getpid()}.{threading.get_ident()}.tmp')


def _save_manifest(m):
    payload = _validate_manifest_payload(dict(m))
    _ensure_stage_dir(STAGING_ABS)
    temp_path = _manifest_temp_path()
    try:
        _write_stage_text(STAGING_ABS, temp_path, json.dumps(payload, indent=2, sort_keys=True))
        _move_stage_file(STAGING_ABS, temp_path, MANIFEST, overwrite=True)
    finally:
        if _stage_exists(temp_path):
            _stage_remove(temp_path)


def manifest_succeeded(entry) -> bool:
    return isinstance(entry, dict) and entry.get('status') == 'succeeded'


def should_refresh_window(win_end, manifest_entry, trailing_cutoff) -> bool:
    if not manifest_succeeded(manifest_entry):
        return True
    completed_at = _as_utc_datetime((manifest_entry or {}).get('completed_at'))
    return _as_utc_datetime(win_end) > _as_utc_datetime(trailing_cutoff) or completed_at is None


def window_stage_files_reusable(base_dir: str, key: str, manifest_entry) -> bool:
    entry = manifest_entry or {}
    recorded = entry.get('files')
    if isinstance(recorded, list):
        if recorded:
            expected = [_stage_path_join(base_dir, _stage_basename(path)) for path in recorded]
            return all(_stage_exists(path) for path in expected)
        return int(entry.get('rows', 0) or 0) == 0 and int(entry.get('pages', 0) or 0) == 0
    files = list_window_files(base_dir, key, include_partial=False)
    if files:
        return True
    return int(entry.get('rows', 0) or 0) == 0 and int(entry.get('pages', 0) or 0) == 0


def is_window_reusable(base_dir: str, key: str, win_end, manifest_entry, trailing_cutoff) -> bool:
    return (not should_refresh_window(win_end, manifest_entry, trailing_cutoff)) and window_stage_files_reusable(
        base_dir, key, manifest_entry
    )


def _read_manifest_entry(key: str) -> dict:
    with MANIFEST_LOCK:
        return dict(manifest.get(key, {}))


def _mark_window(key: str, status: str, **extra):
    if not isinstance(key, str) or not key.strip():
        raise ValueError(f'Window key must be a non-empty string; found {key!r}.')
    if not isinstance(status, str) or status not in _allowed_manifest_statuses():
        raise ValueError(f'Unsupported manifest status {status!r}.')
    with MANIFEST_LOCK:
        # The initial disk snapshot is loaded once. Remote listings or mounted reads can lag a
        # successful publish; reloading here can discard newer checkpoints from other workers.
        # Sharing a staging directory between separate runs remains unsupported.
        current = dict(manifest)
        entry = dict(current.get(key, {}))
        entry.update(extra)
        entry['status'] = status
        current[key] = entry
        _save_manifest(current)
        manifest.clear()
        manifest.update(current)


def _row(r):
    return canonicalize_audit_record(r)


def _is_graph_records_url(url: str, qid: str) -> bool:
    parsed = urlparse(url)
    return (
        parsed.scheme == 'https'
        and parsed.netloc == 'graph.microsoft.com'
        and parsed.path.startswith(f'/beta/security/auditLog/queries/{qid}/records')
    )


def _validate_graph_records_url(url, qid: str) -> str:
    if not isinstance(url, str) or not url.strip():
        raise RuntimeError(f'Query {qid} returned missing/invalid @odata.nextLink: {url!r}')
    url = url.strip()
    if not _is_graph_records_url(url, qid):
        raise RuntimeError(f'Query {qid} returned non-Graph records continuation URL: {url}')
    return url


def _validate_records_page(qid: str, current_url: str, payload):
    if not isinstance(payload, dict):
        raise RuntimeError(
            f'Query {qid} returned malformed records payload type {type(payload).__name__} for {current_url}'
        )
    if 'value' not in payload:
        raise RuntimeError(f'Query {qid} records payload missing value list for {current_url}')
    vals = payload['value']
    if not isinstance(vals, list):
        raise RuntimeError(f'Query {qid} records payload value must be a list for {current_url}')
    for index, record in enumerate(vals):
        if not isinstance(record, dict):
            raise RuntimeError(
                f'Query {qid} records payload item {index} must be an object for {current_url}'
            )
    next_link = payload.get('@odata.nextLink')
    if next_link is None:
        return vals, None
    return vals, _validate_graph_records_url(next_link, qid)


def _drain(qid, key):
    n, page = 0, 0
    partial_files = []
    purge_window_files(STAGING_ABS, key, include_partial=True)
    url = _validate_graph_records_url(
        f'https://graph.microsoft.com/beta/security/auditLog/queries/{qid}/records?$top=999',
        qid,
    )
    seen_urls = set()
    while url:
        if url in seen_urls:
            raise RuntimeError(f'Query {qid} returned repeated pagination link cycle: {url}')
        seen_urls.add(url)
        r = _request('GET', url, headers=get_headers(), timeout=60)
        r.raise_for_status()
        vals, next_url = _validate_records_page(qid, url, r.json())
        if next_url in seen_urls:
            raise RuntimeError(f'Query {qid} returned repeated pagination link cycle: {next_url}')
        if vals:
            part_path = _stage_path_join(STAGING_ABS, f'win_{key}_{page:04d}.jsonl.partial')
            _write_stage_jsonl(STAGING_ABS, part_path, vals)
            partial_files.append(part_path)
            page += 1
            n += len(vals)
        url = next_url
    for part_path in partial_files:
        _move_stage_file(STAGING_ABS, part_path, part_path[:-8], overwrite=True)
    return n, page


def _process(win, attempt=1):
    ws, we = win
    key = stable_window_key(ws, we)
    entry = _read_manifest_entry(key)
    trailing_cutoff = end_date - timedelta(days=max(int(LOOKBACK_DAYS), 0))
    if is_window_reusable(STAGING_ABS, key, we, entry, trailing_cutoff):
        return key, entry.get('rows', 0), 'skip'
    _mark_window(
        key,
        'querying',
        window_start=_as_utc_datetime(ws).isoformat(),
        window_end=_as_utc_datetime(we).isoformat(),
        refreshed_at=datetime.now(timezone.utc).isoformat(),
        attempts=attempt,
        error=None,
        **({'attempt_errors': []} if attempt == 1 else {}),
    )
    qid = None
    try:
        # Only creating and waiting take one of the tenant's concurrent query slots; draining doesn't.
        with QUERY_LIMITER:
            qid = create_query(ws, we)
            _mark_window(key, 'waiting', query_id=qid)
            wait_for_query(qid)
        _mark_window(key, 'draining', query_id=qid)
        n, pages = _drain(qid, key)
        _mark_window(
            key,
            'succeeded',
            query_id=qid,
            rows=n,
            pages=pages,
            files=[_stage_basename(p) for p in list_window_files(STAGING_ABS, key)],
            completed_at=datetime.now(timezone.utc).isoformat(),
        )
        return key, n, 'done'
    except Exception as e:
        failure = {
            'error': f'{type(e).__name__}: {e}',
            'completed_at': None,
        }
        if qid:
            failure['query_id'] = qid
        _mark_window(key, 'failed', **failure)
        purge_window_files(STAGING_ABS, key, include_partial=True)
        raise


_FATAL_HTTP = {401, 403}


def _http_status(exc):
    status = getattr(getattr(exc, 'response', None), 'status_code', None)
    return status if isinstance(status, int) else None


def is_retryable_window_error(exc) -> bool:
    """Failures a fresh query can fix: failed/cancelled by the service, timeouts, throttling, outages."""
    if isinstance(exc, (AuditQueryFailed, TimeoutError, ConnectionError)):
        return True
    status = _http_status(exc)
    if status is not None:
        return status in _TRANSIENT
    return type(exc).__module__.split('.')[0] in ('requests', 'urllib3')


def _run_window(win):
    """Fetch one window: retry with a new query, then split it when the retries run out.

    Returns (key, rows, outcome, detail). outcome is 'skip', 'done', 'split' (detail = the halves) or
    'failed' (detail = the last error). Raises only for 401/403, which would fail every window.
    """
    ws, we = win
    key = stable_window_key(ws, we)
    retries = max(int(WINDOW_RETRIES), 0)
    errors, last = [], None
    for attempt in range(1, retries + 2):
        try:
            _, rows, how = _process(win, attempt=attempt)
            return key, rows, how, None
        except Exception as exc:
            if _http_status(exc) in _FATAL_HTTP:
                raise
            last = f'{type(exc).__name__}: {exc}'
            errors.append(f'attempt {attempt}: {last}')
            _mark_window(key, 'failed', attempts=attempt, attempt_errors=errors[-5:])
            if not is_retryable_window_error(exc):
                return key, 0, 'failed', last
            if QUERY_LIMITER.shrink(f'{key}: {last}', halve=_http_status(exc) == 429):
                print(f'    concurrency lowered to {QUERY_LIMITER.limit} after: {last}')
            if isinstance(exc, TimeoutError) and SPLIT_ON_TIMEOUT:
                break
            if attempt <= retries:
                delay = retry_delay(attempt, RETRY_BASE_SEC, RETRY_MAX_SEC)
                print(f'    {key}: {last}; new query in {delay:.0f}s (retry {attempt}/{retries})')
                time.sleep(delay)
    halves = split_window(ws, we, MIN_CHUNK_HOURS)
    if not halves:
        return key, 0, 'failed', last
    _mark_window(
        key,
        'split',
        parts=[stable_window_key(h_ws, h_we) for h_ws, h_we in halves],
        split_at=datetime.now(timezone.utc).isoformat(),
        error=last,
        completed_at=None,
    )
    purge_window_files(STAGING_ABS, key, include_partial=True)
    return key, 0, 'split', halves


def leaf_windows(planned):
    """The windows that hold the data for a plan: split windows are replaced by their parts."""
    with MANIFEST_LOCK:
        snapshot = dict(manifest)
    return [leaf for ws, we in planned for leaf in expand_split_windows(snapshot, ws, we, MIN_CHUNK_HOURS)]


def _fmt_range(ws, we):
    return f'{_as_utc_datetime(ws):%Y-%m-%d %H:%M} -> {_as_utc_datetime(we):%Y-%m-%d %H:%M} UTC'


def assert_no_unrecovered_gaps(not_before, before):
    gaps = uncovered_failed_ranges(manifest, not_before, before)
    if gaps:
        lines = '\n'.join(f'  - {_fmt_range(s, e)}' for s, e in gaps)
        raise RuntimeError(
            f'{len(gaps)} earlier audit range(s) failed and were never recovered:\n{lines}\n'
            f'This incremental run starts at {_as_utc_datetime(before):%Y-%m-%d %H:%M} UTC, so it would leave '
            f"them empty. Rerun with MODE='backfill' first: succeeded windows are reused, so only these ranges "
            f'are queried again. To accept the gap instead, set ALLOW_UNRECOVERED_GAPS = True.'
        )


def fetch_windows(planned):
    """Fetch every planned window (retrying and splitting); return (active keys, staged rows)."""
    trailing_cutoff = end_date - timedelta(days=max(int(LOOKBACK_DAYS), 0))
    leaves = leaf_windows(planned)
    pending = [
        (ws, we) for ws, we in leaves
        if not is_window_reusable(
            STAGING_ABS, stable_window_key(ws, we), we, _read_manifest_entry(stable_window_key(ws, we)), trailing_cutoff
        )
    ]
    print(
        f'{len(leaves) - len(pending)} window(s) already reusable; {len(pending)} to fetch '
        f'(<= {QUERY_LIMITER.limit} queries at a time; a failing window is retried {WINDOW_RETRIES}x, '
        f'then split down to {MIN_CHUNK_HOURS:g}h).'
    )
    outcomes, fatal, done, jobs = {}, None, 0, len(pending)
    with ThreadPoolExecutor(max_workers=max(1, int(MAX_CONCURRENT_QUERIES))) as ex:
        running = {ex.submit(_run_window, w): w for w in pending}
        while running:
            finished, _ = wait_futures(list(running), return_when=FIRST_COMPLETED)
            for fut in finished:
                running.pop(fut)
                if fut.cancelled():
                    continue
                try:
                    key, n, how, detail = fut.result()
                except Exception as e:
                    fatal = fatal or e
                    for other in running:
                        other.cancel()
                    continue
                outcomes[key] = how
                done += 1
                if how == 'split':
                    hours = (detail[0][1] - detail[0][0]).total_seconds() / 3600
                    print(f'  [{done}/{jobs}] {key} kept failing -> split into {len(detail)} x {hours:g}h')
                    if fatal is None:
                        for half in detail:
                            running[ex.submit(_run_window, half)] = half
                            jobs += 1
                elif how == 'failed':
                    print(f'  [{done}/{jobs}] {key} FAILED: {detail}')
                else:
                    print(f'  [{done}/{jobs}] {key} {how} (+{n:,})')
    if fatal is not None:
        raise fatal

    leaves = leaf_windows(planned)
    with MANIFEST_LOCK:
        snapshot = dict(manifest)
    keys = [stable_window_key(ws, we) for ws, we in leaves]
    failed = [(w, snapshot.get(k) or {}) for w, k in zip(leaves, keys) if not manifest_succeeded(snapshot.get(k))]
    fetched = [k for k in keys if outcomes.get(k) == 'done']
    retried = [k for k in fetched if int((snapshot.get(k) or {}).get('attempts', 1) or 1) > 1]
    split = sum(1 for how in outcomes.values() if how == 'split')
    rows = sum(int((snapshot.get(k) or {}).get('rows', 0) or 0) for k in keys if manifest_succeeded(snapshot.get(k)))
    shrinks = ', '.join(f'{old}->{new}' for old, new, _ in QUERY_LIMITER.history) or 'unchanged'
    print(
        f'Summary: {len(planned)} planned window(s) -> {len(leaves)} after splits | '
        f'reused {len(leaves) - len(fetched) - len(failed)}, fetched {len(fetched)} ({len(retried)} after a retry), '
        f'split {split}, failed {len(failed)} | concurrency {shrinks} | staged records {rows:,}'
    )
    if failed:
        lines = '\n'.join(
            f'  - {_fmt_range(ws, we)} [{stable_window_key(ws, we)}]: {entry.get("error") or entry.get("status")}'
            for (ws, we), entry in failed
        )
        rerun = "rerun with MODE='backfill'" if MODE == 'backfill' else 'rerun the notebook'
        raise RuntimeError(
            f'{len(failed)} audit window(s) still failed after {WINDOW_RETRIES} retries and splitting down to '
            f'{MIN_CHUNK_HOURS:g}h. Nothing was written to {OUTPUT_TABLE}.\n{lines}\n'
            f'Next step: {rerun}; succeeded windows are reused, so only the windows above are queried again. '
            f'If they keep failing, check for other audit searches running in the tenant and lower '
            f'MAX_CONCURRENT_QUERIES.'
        )
    return set(keys), rows


with MANIFEST_LOCK:
    manifest = _load_manifest()
if MODE == 'incremental' and not ALLOW_UNRECOVERED_GAPS:
    assert_no_unrecovered_gaps(end_date - timedelta(days=max(int(BACKFILL_DAYS), 0)), start_date)
QUERY_LIMITER = AdaptiveLimiter(MAX_CONCURRENT_QUERIES, MIN_CONCURRENT_QUERIES)
THROTTLE_LISTENERS[:] = [lambda: QUERY_LIMITER.shrink('HTTP 429 (throttled) from Graph', halve=True)]
ACTIVE_WINDOW_KEYS, total = fetch_windows(windows)
print(f'Prepared {len(ACTIVE_WINDOW_KEYS)} active window(s); staged files will be read only from this run scope. Manifest: {MANIFEST}')


## 6. Read the staged records (distributed)

Load the staged JSON-lines with Spark — a distributed read, not a driver-side
`createDataFrame` from a giant Python list.

In [ ]:
import os
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, ArrayType

_cols = ['RecordId', 'CreationDate', 'RecordType', 'Operation',
         'AuditData', 'SourceRecordKey', 'AssociatedAdminUnits', 'AssociatedAdminUnitsNames']

# Read only the per-window JSONL files that belong to the current window set. This prevents
# historical staging files from being re-appended on later incremental runs.
_files = list_active_stage_files(STAGING_ABS, ACTIVE_WINDOW_KEYS)
if not _files:
    _empty = StructType([StructField(c, StringType()) for c in _cols])
    raw = spark.createDataFrame([], _empty)
else:
    raw = spark.read.json(_files)

raw = raw.persist()
print(f'Raw DataFrame rows: {raw.count():,} (from {len(_files)} active window file(s))')


## 7. Parse `AuditData` JSON

Schema mirrors the existing `Copilot_Audit_Log_Parser.ipynb` — only the fields the PBIT consumes.


In [ ]:
audit_schema = StructType([
    StructField('CreationTime', StringType()),
    StructField('UserId', StringType()),
    StructField('Workload', StringType()),
    StructField('ApplicationName', StringType()),
    StructField('ClientRegion', StringType()),
    StructField('AgentId', StringType()),
    StructField('AgentName', StringType()),
    # Copilot Studio runtime records: AgentPlatform 'CopilotStudio' and
    # PlatformAgentId '<environment>_<bot id>'. They usually carry no Messages.
    StructField('AgentPlatform', StringType()),
    StructField('PlatformAgentId', StringType()),
    StructField('ConversationId', StringType()),
    StructField('AppIdentity', StructType([
        StructField('AppId', StringType()),
        StructField('DisplayName', StringType()),
        StructField('PublisherId', StringType()),
    ])),
    StructField('CopilotEventData', StructType([
        StructField('AppHost', StringType()),
        StructField('ThreadId', StringType()),
        StructField('ConversationId', StringType()),
        StructField('SensitivityLabelId', StringType()),
        StructField('Contexts', ArrayType(StructType([
            StructField('Type', StringType())
        ]))),
        StructField('AISystemPlugin', ArrayType(StructType([
            StructField('Id', StringType()),
            StructField('Name', StringType())
        ]))),
        StructField('ModelTransparencyDetails', ArrayType(StructType([
            StructField('ModelName', StringType())
        ]))),
        StructField('AccessedResources', ArrayType(StructType([
            StructField('Type', StringType()),
            StructField('Action', StringType()),
            StructField('SiteUrl', StringType()),
            StructField('SensitivityLabelId', StringType()),
            StructField('_StableKey', StringType()),
            StructField('_StableOrdinal', StringType()),
        ]))),
        StructField('Messages', ArrayType(StructType([
            StructField('Id', StringType()),
            StructField('isPrompt', StringType()),
            StructField('_StableKey', StringType()),
            StructField('_StableOrdinal', StringType()),
        ]))),
    ])),
])

parsed = (raw
    .filter(F.col('AuditData').isNotNull() & (F.length('AuditData') > 10))
    .withColumn('j', F.from_json('AuditData', audit_schema))
    .filter(F.col('j').isNotNull()))

## 8. Flatten + fan out per (prompt × resource)

Mirrors the M-query exactly: drop responses, expand resources. No-prompt Copilot Studio runtime and Cowork autonomous records are kept as one task row per record.


In [ ]:
_empty_message = F.struct(
    F.lit(None).cast(StringType()).alias('Id'),
    F.lit(None).cast(StringType()).alias('isPrompt'),
    F.lit(None).cast(StringType()).alias('_StableKey'),
    F.lit(None).cast('long').alias('_StableOrdinal'),
)
_empty_resource = F.struct(
    F.lit(None).cast(StringType()).alias('Type'),
    F.lit(None).cast(StringType()).alias('Action'),
    F.lit(None).cast(StringType()).alias('SiteUrl'),
    F.lit(None).cast(StringType()).alias('SensitivityLabelId'),
    F.lit('resource:none').cast(StringType()).alias('_StableKey'),
    F.lit(0).cast('long').alias('_StableOrdinal'),
)


def _nonblank(col):
    return F.length(F.trim(F.coalesce(col.cast('string'), F.lit('')))) > 0


# Microsoft documents AppIdentity as a 'workload.appGroup.appName' string (for example
# 'Copilot.Studio.<AppId>' or 'MicrosoftAgent.<name>.P_<id>'). The struct fields above
# stay null for that shape, so keep the raw text for the agent-type rules. An object
# value falls back to its DisplayName.
_app_identity_json = F.get_json_object(F.col('AuditData'), '$.AppIdentity')
_app_identity_text = (
    F.when(F.trim(_app_identity_json).startswith('{'), F.col('j.AppIdentity.DisplayName'))
     .otherwise(_app_identity_json)
)


def _ced_text(field):
    return F.get_json_object(F.col('AuditData'), f'$.CopilotEventData.{field}')


_base = (
    parsed
    .select(
        F.col('RecordId'),
        F.col('Operation'),
        F.col('AuditData').alias('_AuditDataRaw'),
        F.col('SourceRecordKey').alias('_SourceRecordKey'),
        F.col('j.CreationTime').cast('timestamp').alias('CreationDate'),
        F.col('j.AgentId').alias('AgentId'),
        F.col('j.AgentName').alias('AgentName'),
        F.col('j.AgentPlatform').alias('AgentPlatform'),
        F.col('j.PlatformAgentId').alias('PlatformAgentId'),
        F.col('j.AppIdentity.AppId').alias('AppIdentity_AppId'),
        F.col('j.AppIdentity.DisplayName').alias('AppIdentity_DisplayName'),
        F.col('j.AppIdentity.PublisherId').alias('AppIdentity_PublisherId'),
        _app_identity_text.alias('AppIdentity_Text'),
        # Agent fields current exports carry inside CopilotEventData (agent-type rules).
        _ced_text('TargetPlatformAgentId').alias('Agent_TargetPlatformId'),
        F.coalesce(_ced_text('TargetAgentName'), _ced_text('AgentName')).alias('Agent_TargetName'),
        _ced_text('PlatformAgentType').alias('Agent_PlatformType'),
        F.col('j.ApplicationName').alias('ApplicationName'),
        F.col('j.ClientRegion').alias('ClientRegion'),
        F.col('j.UserId').alias('Audit_UserId'),
        F.lower(F.trim(F.col('j.UserId'))).alias('Audit_UserId_Normalized'),
        F.col('j.Workload').alias('Workload'),
        F.col('j.CopilotEventData.AppHost').alias('AppHost'),
        # Runtime records name the conversation ConversationId rather than ThreadId.
        F.coalesce(
            F.when(_nonblank(F.col('j.CopilotEventData.ThreadId')), F.col('j.CopilotEventData.ThreadId')),
            F.when(_nonblank(F.col('j.CopilotEventData.ConversationId')), F.col('j.CopilotEventData.ConversationId')),
            F.when(_nonblank(F.col('j.ConversationId')), F.col('j.ConversationId')),
        ).alias('ThreadId'),
        F.col('j.CopilotEventData.SensitivityLabelId').alias('SensitivityLabelId'),
        F.col('j.CopilotEventData.Contexts')[0]['Type'].alias('Context_Type'),
        F.col('j.CopilotEventData.AISystemPlugin')[0]['Id'].alias('AISystemPlugin_Id'),
        F.col('j.CopilotEventData.AISystemPlugin')[0]['Name'].alias('AISystemPlugin_Name'),
        F.col('j.CopilotEventData.ModelTransparencyDetails')[0]['ModelName']
            .alias('ModelTransparencyDetails_ModelName'),
        F.col('j.CopilotEventData.AccessedResources').alias('Resources'),
        F.col('j.CopilotEventData.Messages').alias('Messages'),
        F.size('j.CopilotEventData.AccessedResources').alias('Resource_Count'),
    )
)

_ambiguous_synthetic_records = (
    _base
    .filter(~_nonblank(F.col('RecordId')) & _nonblank(F.col('_SourceRecordKey')))
    .groupBy('_SourceRecordKey')
    .count()
    .filter(F.col('count') > 1)
    .limit(1)
    .collect()
)
if _ambiguous_synthetic_records:
    raise RuntimeError(
        "Missing RecordId produced ambiguous synthetic SourceRecordKey values; refusing unsafe dedup."
    )

# A record with no prompt message has no prompt to count. Copilot Studio runtime
# records (agents used in Teams, websites and other channels) and Microsoft 365
# Copilot Cowork scheduled/autonomous runs are the exceptions: they are the only
# audit trail of that usage, so they are kept as one task row per record with
# Prompts_Available = 'FALSE'. Other no-prompt records are still dropped.
_has_messages = F.coalesce(F.size('Messages'), F.lit(-1)) > 0
_has_prompt_messages = F.coalesce(
    F.expr("exists(Messages, m -> lower(cast(m.isPrompt as string)) = 'true')"),
    F.lit(False),
)
_copilot_studio_runtime = (
    F.lower(F.regexp_replace(F.coalesce(F.col('AgentPlatform'), F.lit('')), '\\s', '')) == 'copilotstudio'
)


def is_cowork_autonomous(app_host_col, app_identity_col):
    _app_host = F.lower(F.trim(F.coalesce(app_host_col.cast('string'), F.lit(''))))
    _app_identity = F.lower(F.trim(F.coalesce(app_identity_col.cast('string'), F.lit(''))))
    return (_app_host == 'cowork') | _app_identity.startswith('copilot.m365copilot.cowork')


_cowork_autonomous = is_cowork_autonomous(F.col('AppHost'), F.col('AppIdentity_Text'))
_task_row_record = _copilot_studio_runtime | _cowork_autonomous

flat = (
    _base
    .withColumn('Prompts_Available', F.when(_has_prompt_messages, F.lit('TRUE')).otherwise(F.lit('FALSE')))
    .withColumn('_Task_Row_Placeholder', (F.col('Prompts_Available') == 'FALSE') & _task_row_record)
    .withColumn(
        'Source_RecordKey',
        F.when(_nonblank(F.col('_SourceRecordKey')), F.trim(F.col('_SourceRecordKey').cast('string')))
         .when(_nonblank(F.col('RecordId')), F.concat(F.lit('rid:'), F.trim(F.col('RecordId').cast('string'))))
    )
    .select(
        '*',
        F.posexplode_outer(
            F.when(F.col('_Task_Row_Placeholder'), F.array(_empty_message))
             .when(F.size('Messages') > 0, F.col('Messages'))
             .otherwise(F.array(_empty_message))
        ).alias('Message_ArrayOrdinal', 'msg'),
    )
    .filter(
        (F.lower(F.col('msg.isPrompt').cast('string')) == 'true')
        | F.col('_Task_Row_Placeholder')
    )
    .select(
        '*',
        F.posexplode_outer(
            F.when(F.col('_Task_Row_Placeholder'), F.array(_empty_resource))
             .when(F.size('Resources') > 0, F.col('Resources'))
             .otherwise(F.array(_empty_resource))
        ).alias('Resource_ArrayOrdinal', 'res'),
    )
    .withColumn(
        'Source_MessageKey',
        F.coalesce(
            F.trim(F.col('msg._StableKey').cast('string')),
            F.when(F.col('_Task_Row_Placeholder'), F.lit('message:none')),
        ),
    )
    .withColumn('Source_ResourceKey', F.trim(F.col('res._StableKey').cast('string')))
    .withColumn(
        'Message_Ordinal',
        F.coalesce(F.col('msg._StableOrdinal').cast('long'), F.col('Message_ArrayOrdinal').cast('long')),
    )
    .withColumn(
        'Resource_Ordinal',
        F.coalesce(F.col('res._StableOrdinal').cast('long'), F.col('Resource_ArrayOrdinal').cast('long'), F.lit(0).cast('long')),
    )
)

_missing_identity_keys = (
    flat
    .filter(
        ~_nonblank(F.col('Source_RecordKey'))
        | ~_nonblank(F.col('Source_MessageKey'))
        | ~_nonblank(F.col('Source_ResourceKey'))
    )
    .limit(1)
    .collect()
)
if _missing_identity_keys:
    raise RuntimeError(
        "Canonical audit identity fields are missing from staged data; rerun ingestion with the canonical stage helpers."
    )

flat = (
    flat
    .withColumn(
        'Id',
        F.sha2(
            F.concat_ws(
                '||',
                F.col('Source_RecordKey'),
                F.col('Source_MessageKey'),
                F.col('Source_ResourceKey'),
            ),
            256,
        ),
    )
    .select(
        'Id', 'RecordId', 'Source_RecordKey', 'Source_MessageKey', 'Source_ResourceKey',
        'CreationDate', 'AgentId', 'AgentName', 'AgentPlatform', 'PlatformAgentId',
        'AppIdentity_AppId', 'AppIdentity_DisplayName', 'AppIdentity_PublisherId',
        'AppIdentity_Text', 'Agent_TargetPlatformId', 'Agent_TargetName', 'Agent_PlatformType',
        'ApplicationName', 'ClientRegion',
        'Audit_UserId', 'Audit_UserId_Normalized', 'Workload',
        'AppHost', 'ThreadId', 'SensitivityLabelId',
        'Context_Type',
        'AISystemPlugin_Id', 'AISystemPlugin_Name',
        'ModelTransparencyDetails_ModelName',
        F.col('res.Type').alias('AccessedResource_Type'),
        F.col('res.Action').alias('AccessedResource_Action'),
        F.col('res.SiteUrl').alias('AccessedResource_SiteUrl'),
        F.col('res.SensitivityLabelId').alias('AccessedResource_SensitivityLabelId'),
        F.when(F.col('_Task_Row_Placeholder'), F.lit('message:none'))
         .otherwise(F.col('msg.Id')).alias('Message_Id'),
        F.when(F.col('_Task_Row_Placeholder'), F.lit('false'))
         .otherwise(F.col('msg.isPrompt')).alias('Message_isPrompt'),
        'Prompts_Available',
        F.col('Message_Ordinal').cast('long').alias('Message_Ordinal'),
        F.col('Resource_Ordinal').cast('long').alias('Resource_Ordinal'),
        F.coalesce(F.col('Resource_Count'), F.lit(1)).cast('long').alias('Resource_Count'),
        F.to_date('CreationDate').alias('InteractionDate'),
        F.expr("date_trunc('week', CreationDate)").cast('date').alias('WeekStart'),
        F.expr("date_trunc('month', CreationDate)").cast('date').alias('MonthStart'),
    )
    .dropDuplicates(['Id'])
)

## 9. Derive `Agent_TitleID` + `Agent_EntraId`

`Agent_TitleID` keeps the legacy Copilot Studio logic (existing parser + M-query).
`Agent_EntraId` captures the **Microsoft Entra Agent ID** GUID that Agent 365 now
stamps into the audit `AgentId` (instead of the `CopilotStudio.Declarative.{title}`
string). Without it, Entra-registered agents fall through to NULL and drop out of
the Agents join. Both keys are emitted so old and new agents resolve via the
registry crosswalk; neither overwrites the other.


In [ ]:
flat = flat.withColumn('Agent_TitleID', F.expr(r'''
    CASE
      WHEN AgentId IS NULL THEN NULL
      WHEN AgentId LIKE '%CopilotStudio.Declarative.%' THEN
           split(split(AgentId, 'CopilotStudio\\.Declarative\\.')[1], '\\.')[0]
      WHEN AgentId LIKE 'P\\_%' OR AgentId LIKE 'T\\_%' THEN
           split(AgentId, '\\.')[0]
      ELSE NULL
    END
'''))

# Microsoft Entra Agent ID (Agent 365): the audit AgentId now carries an Entra
# Agent ID GUID rather than the CopilotStudio.Declarative.{title} string, so the
# legacy CASE above lands NULL. Capture the GUID only when no legacy Title ID was
# parsed, so a GUID embedded inside a declarative title is never mistaken for it.
flat = flat.withColumn('Agent_EntraId', F.expr(r'''
    CASE
      WHEN Agent_TitleID IS NOT NULL THEN NULL
      WHEN AgentId IS NULL THEN NULL
      ELSE nullif(
             regexp_extract(
               AgentId,
               '[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}',
               0),
             '')
    END
'''))
# An all-zero AgentId (Fabric multi-agent orchestration) is not an agent identity.
flat = flat.withColumn(
    'Agent_EntraId',
    F.when(F.regexp_replace(F.coalesce(F.col('Agent_EntraId'), F.lit('')), '[-0]', '') == '', F.lit(None).cast(StringType()))
     .otherwise(F.col('Agent_EntraId')),
)

# Copilot Studio runtime records carry the bot, not the Title ID:
# PlatformAgentId = '<environment id>_<bot id>'. A bare GUID is taken as the bot id.
PLATFORM_AGENT_ID_PATTERN = r'^(.*)_([0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12})$'
BARE_GUID_PATTERN = r'^[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}$'
_platform_agent_id = F.trim(F.coalesce(F.col('PlatformAgentId'), F.lit('')))
flat = (flat
    .withColumn('Agent_BotId', F.lower(
        F.when(_platform_agent_id.rlike(PLATFORM_AGENT_ID_PATTERN),
               F.regexp_extract(_platform_agent_id, PLATFORM_AGENT_ID_PATTERN, 2))
         .when(_platform_agent_id.rlike(BARE_GUID_PATTERN), _platform_agent_id)))
    .withColumn('Agent_EnvironmentId',
        F.when(_platform_agent_id.rlike(PLATFORM_AGENT_ID_PATTERN),
               F.regexp_extract(_platform_agent_id, PLATFORM_AGENT_ID_PATTERN, 1))))

# Records that are not end-user agent usage. Flagged with the first matching reason:
#   AppHost "Copilot Studio"       -> the maker's test pane
#   AppHost "pva-maker-evaluation" -> Copilot Studio evaluation runs
#   AppHost "agentic-builder"      -> agent authoring in Agent Builder
#   AppHost "autonomous"           -> autonomous (trigger-driven) agent runs
#   AppHost "workflow-agents"      -> workflow agent runs
#   no messages + AppHost "M365Copilot" -> the Copilot Studio runtime twin of an
#       M365 Copilot chat that is already counted from its own prompt record
#   all-zero AgentId               -> Fabric multi-agent orchestration
# Plain SQL (lower/trim/coalesce/replace) so the same text runs in Spark and in the tests.
EXCLUDE_REASON_SQL = """
CASE
  WHEN lower(trim(coalesce(AppHost, ''))) = 'copilot studio' THEN 'Copilot Studio test pane'
  WHEN lower(trim(coalesce(AppHost, ''))) = 'pva-maker-evaluation' THEN 'Maker evaluation'
  WHEN lower(trim(coalesce(AppHost, ''))) = 'agentic-builder' THEN 'Agent authoring'
  WHEN lower(trim(coalesce(AppHost, ''))) = 'autonomous' THEN 'Autonomous run'
  WHEN lower(trim(coalesce(AppHost, ''))) = 'workflow-agents' THEN 'Workflow run'
  WHEN upper(trim(coalesce(Prompts_Available, 'TRUE'))) = 'FALSE'
       AND lower(trim(coalesce(AppHost, ''))) = 'm365copilot' THEN 'M365 Copilot twin'
  WHEN trim(coalesce(AgentId, '')) <> ''
       AND replace(replace(lower(trim(coalesce(AgentId, ''))), '-', ''), '0', '') = ''
       THEN 'Fabric multi-agent'
  ELSE NULL
END
"""

# Flag only; the processor drops the reasons listed in its DROP_EXCLUDE_REASONS.
flat = flat.withColumn('Exclude_Reason', F.expr(EXCLUDE_REASON_SQL))

## 10. Write to Lakehouse Delta table


In [ ]:
def _validate_batch_keys(df, key_columns, dataset_name: str):
    missing = [c for c in key_columns if c not in df.columns]
    if missing:
        raise RuntimeError(
            f'{dataset_name} is missing stable key columns {missing}; cannot perform idempotent incremental merge.'
        )
    has_blank = (
        df.where(F.col('Id').isNull() | (F.trim(F.col('Id').cast('string')) == ''))
          .limit(1)
          .count()
    )
    if has_blank:
        raise RuntimeError(
            f'{dataset_name} contains blank Id values; rerun after fixing key extraction rather than appending duplicates.'
        )

def write_output(df):
    if WRITE_MODE not in ('overwrite', 'merge'):
        raise ValueError(f'Unsupported WRITE_MODE={WRITE_MODE!r}')
    if WRITE_MODE == 'merge':
        _validate_batch_keys(df, KEY_COLUMNS, 'current parsed batch')
        if spark.catalog.tableExists(OUTPUT_TABLE):
            validate_incremental_target_columns(spark.table(OUTPUT_TABLE).columns, KEY_COLUMNS, OUTPUT_TABLE)
            from delta.tables import DeltaTable
            # New columns (e.g. Agent_BotId, Exclude_Reason) reach an existing table only
            # with schema auto-merge; otherwise updateAll/insertAll silently drop them.
            spark.conf.set('spark.databricks.delta.schema.autoMerge.enabled', 'true')
            tgt = DeltaTable.forName(spark, OUTPUT_TABLE)
            (tgt.alias('t').merge(df.alias('s'), 't.`Id` = s.`Id`')
                .whenMatchedUpdateAll()
                .whenNotMatchedInsertAll()
                .execute())
            return 'merge'
    (df.write
        .format('delta')
        .mode('overwrite')
        .option('overwriteSchema', 'true')
        .saveAsTable(OUTPUT_TABLE))
    return 'overwrite'

write_strategy = write_output(flat)
row_count = spark.table(OUTPUT_TABLE).count()
print(f'✓ Rows written to {OUTPUT_TABLE}: {row_count:,} (strategy={write_strategy})')

## 11. Verify

Spot-check the output. Expect populated `AISystemPlugin_Id` (e.g. `BingWebSearch`), `Workload`, `AppHost`.


In [ ]:
spark.table(OUTPUT_TABLE).select(
    'AISystemPlugin_Id', 'Workload', 'AppHost'
).groupBy('AISystemPlugin_Id', 'Workload', 'AppHost').count().orderBy(F.desc('count')).show(20, truncate=False)


---
**Connect the PBIT**: open the Fabric variant of either dashboard and supply the **Fabric SQL Endpoint** + **Lakehouse Database** parameters as before. The PBIT reads `Copilot_Interactions_Parsed` directly via the SQL endpoint — no other parameter changes needed when switching from the CSV-based parser to this direct ingester.
